# Step 3 - Feature engineering

Build pre-post content/context features for every labelled video and join them to both targets
(virality + sentiment). Outcome signals (views/likes/comments/comment-sentiment) are NOT used as
features to avoid label leakage.

In [13]:
# load 2 label and video tables
from pathlib import Path
import pandas as pd
import numpy as np
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
SAMPLES = ROOT / "data" / "samples"

labels = pd.read_parquet(ROOT / "ml" / "data" / "video_labels.parquet")[
    ["video_id", "virality_score", "virality_class", "virality_label",
     "sentiment_score", "sentiment_class", "sentiment_label"]
]
videos = pd.read_csv(SAMPLES / "video_context_dataset.csv", parse_dates=["video_published_at"])

data = videos.merge(labels, on="video_id", how="inner")   # only labelled videos
print("rows:", data.shape[0])
print(data["virality_label"].value_counts())
print(data["sentiment_label"].value_counts())

rows: 515
virality_label
high      172
low       172
medium    171
Name: count, dtype: int64
sentiment_label
negative    172
positive    172
neutral     171
Name: count, dtype: int64


In [14]:
# text normalization + missing data flags
for col in ["video_title", "video_description", "video_transcript"]:
    data[col] = data[col].fillna("").astype(str)

data["has_description"] = (data["video_description"].str.len() > 0).astype(int)
data["has_transcript"]  = (data["video_transcript"].str.len() > 0).astype(int)

# combined raw text -> TF-IDF will be built inside the model pipeline (Step 4) to avoid leakage
data["text_all"] = (data["video_title"] + " . " +
                    data["video_description"] + " . " +
                    data["video_transcript"]).str.strip()

In [15]:
# pre-post features only
sia = SentimentIntensityAnalyzer()

# text length
data["title_len_words"] = data["video_title"].str.split().str.len()
data["desc_len_words"]  = data["video_description"].str.split().str.len()
data["trans_len_words"] = data["video_transcript"].str.split().str.len()

# title tone & style
data["title_sentiment"]    = data["video_title"].map(lambda t: sia.polarity_scores(t)["compound"])
data["title_has_question"] = data["video_title"].str.contains(r"\?", regex=True).astype(int)
data["title_upper_ratio"]  = data["video_title"].map(
    lambda t: sum(c.isupper() for c in t) / max(len(t), 1))

# EV domain keywords from the post's OWN text (title + description), independent of transcript
# (avoids entangling keyword flags with transcript availability)
text_lower = (data["video_title"] + " " + data["video_description"]).str.lower()
data["kw_price"]    = text_lower.str.contains(r"price|cost|\$|cheap|expensive").astype(int)
data["kw_range"]    = text_lower.str.contains(r"range|mile|miles|km").astype(int)
data["kw_charging"] = text_lower.str.contains(r"charg|battery|kwh").astype(int)

# publish timing
dt = data["video_published_at"].dt
data["pub_hour"]  = dt.hour
data["pub_dow"]   = dt.dayofweek
data["pub_month"] = dt.month

# duration
data["duration_min"] = data["video_duration_seconds"] / 60.0

# channel frequency = reputation/activity proxy, known before posting (NOT an outcome)
ch_freq = data["video_channel"].value_counts()
data["channel_freq"] = data["video_channel"].map(ch_freq)

## Cognitive friction (new feature)

Reading-effort score in [0, 1] from `title + description` only (pre-launch text, no transcript). Higher = harder to read. Adds the composite `cognitive_friction_score` plus five sub-signals (`f_word / f_sent / f_clause / f_info / f_visual`) so the model and the explanation layer can see *why* friction is high.

In [ ]:
# cognitive friction: reading-effort score from the pre-launch text
import sys
sys.path.insert(0, str(ROOT / "ml"))
from features.cognitive_friction import cognitive_friction

friction_text = (data["video_title"] + ". " + data["video_description"]).fillna("")
friction = friction_text.map(cognitive_friction).apply(pd.Series)
data = pd.concat([data.drop(columns=friction.columns, errors="ignore"), friction], axis=1)

# length sibling: friction is ~0.68 correlated with text length (B5), so expose length
# too — this lets the tree model isolate friction's length-independent effect.
data["content_len_words"] = friction_text.str.split().str.len()

print(data["cognitive_friction_score"].describe().round(3))
data[["video_title", "cognitive_friction_score"]].head()

In [16]:
# combine features for modeling
num_features = [
    "title_len_words", "desc_len_words", "trans_len_words",
    "title_sentiment", "title_has_question", "title_upper_ratio",
    "kw_price", "kw_range", "kw_charging",
    "pub_hour", "pub_dow", "pub_month",
    "duration_min", "channel_freq", "has_description", "has_transcript",
    "cognitive_friction_score", "f_word", "f_sent", "f_clause", "f_info", "f_visual",
    "content_len_words",
]
label_cols = ["virality_score", "virality_class", "virality_label",
              "sentiment_score", "sentiment_class", "sentiment_label"]

model_df = data[["video_id", "text_all"] + num_features + label_cols].copy()
print("feature table:", model_df.shape)
print(model_df[num_features].describe().round(2).T)

out = ROOT / "ml" / "data" / "video_features.parquet"
model_df.to_parquet(out, index=False)
print("Saved:", out)

feature table: (515, 24)
                    count     mean      std   min     25%      50%      75%  \
title_len_words     515.0    10.73     3.61  3.00    8.00    11.00    13.00   
desc_len_words      515.0   155.59   153.45  0.00   29.50   116.00   230.50   
trans_len_words     515.0  1948.18  2333.48  0.00  125.00  1367.00  3127.00   
title_sentiment     515.0     0.08     0.39 -0.87    0.00     0.00     0.36   
title_has_question  515.0     0.26     0.44  0.00    0.00     0.00     1.00   
title_upper_ratio   515.0     0.17     0.11  0.00    0.11     0.16     0.20   
kw_price            515.0     0.36     0.48  0.00    0.00     0.00     1.00   
kw_range            515.0     0.39     0.49  0.00    0.00     0.00     1.00   
kw_charging         515.0     0.43     0.50  0.00    0.00     0.00     1.00   
pub_hour            515.0    13.61     5.39  0.00   11.00    14.00    17.00   
pub_dow             515.0     2.81     1.96  0.00    1.00     3.00     4.00   
pub_month           515.0  